In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'new-plant-diseases-dataset' dataset.
Path to dataset files: /kaggle/input/new-plant-diseases-dataset


In [ ]:



import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt


In [ ]:
DATASET_DIR = "/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)"
import os

TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VAL_DIR   = os.path.join(DATASET_DIR, "valid")

print("Train exists:", os.path.exists(TRAIN_DIR))
print("Valid exists:", os.path.exists(VAL_DIR))


Train exists: True
Valid exists: True


In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)


In [ ]:
val_datagen = ImageDataGenerator(rescale=1./255)


In [ ]:
import os

os.listdir("/kaggle/input/new-plant-diseases-dataset")


['New Plant Diseases Dataset(Augmented)',
 'new plant diseases dataset(augmented)',
 'test']

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32


In [ ]:
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)


Found 70295 images belonging to 38 classes.
Found 17572 images belonging to 38 classes.


In [ ]:
print(train_generator.class_indices)


{'Apple___Apple_scab': 0, 'Apple___Black_rot': 1, 'Apple___Cedar_apple_rust': 2, 'Apple___healthy': 3, 'Blueberry___healthy': 4, 'Cherry_(including_sour)___Powdery_mildew': 5, 'Cherry_(including_sour)___healthy': 6, 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 7, 'Corn_(maize)___Common_rust_': 8, 'Corn_(maize)___Northern_Leaf_Blight': 9, 'Corn_(maize)___healthy': 10, 'Grape___Black_rot': 11, 'Grape___Esca_(Black_Measles)': 12, 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)': 13, 'Grape___healthy': 14, 'Orange___Haunglongbing_(Citrus_greening)': 15, 'Peach___Bacterial_spot': 16, 'Peach___healthy': 17, 'Pepper,_bell___Bacterial_spot': 18, 'Pepper,_bell___healthy': 19, 'Potato___Early_blight': 20, 'Potato___Late_blight': 21, 'Potato___healthy': 22, 'Raspberry___healthy': 23, 'Soybean___healthy': 24, 'Squash___Powdery_mildew': 25, 'Strawberry___Leaf_scorch': 26, 'Strawberry___healthy': 27, 'Tomato___Bacterial_spot': 28, 'Tomato___Early_blight': 29, 'Tomato___Late_blight': 30, 'Tomato

In [ ]:
NUM_CLASSES = train_generator.num_classes
print("Number of Classes:", NUM_CLASSES)


Number of Classes: 38


In [ ]:

model = models.Sequential([

    layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(256, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(NUM_CLASSES, activation='softmax')
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 24, 24, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 38)             │         9,766 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 465,894 (1.78 MB)

 Trainable params: 464,934 (1.77 MB)

 Non-trainable params: 960 (3.75 KB)

In [ ]:
EPOCHS = 10

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 1194s 538ms/step - accuracy: 0.5229 - loss: 1.6583 - val_accuracy: 0.4322 - val_loss: 2.5785
Epoch 2/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 921s 419ms/step - accuracy: 0.8525 - loss: 0.4664 - val_accuracy: 0.7628 - val_loss: 0.8235
Epoch 3/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 909s 414ms/step - accuracy: 0.9048 - loss: 0.2998 - val_accuracy: 0.8683 - val_loss: 0.4263
Epoch 4/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 916s 417ms/step - accuracy: 0.9279 - loss: 0.2239 - val_accuracy: 0.8193 - val_loss: 0.7518
Epoch 5/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 910s 414ms/step - accuracy: 0.9397 - loss: 0.1910 - val_accuracy: 0.9032 - val_loss: 0.3051
Epoch 6/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 910s 414ms/step - accuracy: 0.9480 - loss: 0.1627 - val_accuracy: 0.9561 - val_loss: 0.1303
Epoch 7/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 914s 416ms/step - accuracy: 0.9544 - loss: 0.1428 - val_accuracy: 0.9231 - val_loss: 0.2586
Epoch 8/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 925s 421ms/step - a

In [ ]:
model.save("plant_disease_custom_cnn.h5")


In [ ]:
import gradio as gr
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.efficientnet import preprocess_input
from PIL import Image
import os

# ==========================
# CONFIG
# ==========================
IMG_SIZE = 224
MODEL_PATH = "plant_disease_efficientnet.h5"

# Path used during training (IMPORTANT)
DATASET_DIR = "dataset/train"   # change if different

# ==========================
# LOAD CLASS NAMES (AUTO)
# ==========================
CLASS_NAMES = sorted(os.listdir(DATASET_DIR))

NUM_CLASSES = len(CLASS_NAMES)
print("Loaded classes:", NUM_CLASSES)

# ==========================
# LOAD MODEL
# ==========================
model = tf.keras.models.load_model(MODEL_PATH)

# ==========================
# PREDICTION FUNCTION
# ==========================
def predict_disease(img):
    if img is None:
        return None, "⚠️ Please upload a leaf image."

    img = img.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    arr = image.img_to_array(img)
    arr = np.expand_dims(arr, axis=0)
    arr = preprocess_input(arr)

    preds = model.predict(arr, verbose=0)[0]

    idx = np.argmax(preds)
    confidence = preds[idx] * 100
    label = CLASS_NAMES[idx]

    result = f"""
### 🧠 Prediction Result
**Disease Class:** `{label}`
**Confidence:** `{confidence:.2f}%`
"""

    return img, result

# ==========================
# CUSTOM CSS
# ==========================
CUSTOM_CSS = """
.container {
    max-width: 1100px;
    margin: auto;
}
h1 {
    font-size: 45px !important;
    font-weight: 700;
}
button {
    font-size: 22px !important;
}
"""

# ==========================
# UI
# ==========================
with gr.Blocks(title="Plant Disease Recognition System") as demo:

    gr.Markdown(
        """
        <div style="text-align:center;">
            <h1>🌱 Plant Disease Recognition System</h1>
            <p>Supports 28 Plant Disease Classes (Deep Learning)</p>
        </div>
        """
    )

    with gr.Row():

        with gr.Column(scale=1):
            image_input = gr.Image(
                type="pil",
                label="Upload Leaf Image",
                height=300
            )

            predict_btn = gr.Button("🔍 Predict Disease")

        with gr.Column(scale=1):
            image_display = gr.Image(
                label="Uploaded Image",
                height=300
            )

            result_output = gr.Markdown()

    predict_btn.click(
        fn=predict_disease,
        inputs=image_input,
        outputs=[image_display, result_output]
    )

# ==========================
# RUN APP
# ==========================
demo.launch(
    theme=gr.themes.Soft(),
    css=CUSTOM_CSS
)


FileNotFoundError: [Errno 2] No such file or directory: 'dataset/train'